# Pesquisa de Subenquadramento de Engenharia — PNCP

Modelo para identificar contratos rotulados como **"serviços gerais"** que na
verdade são **engenharia/obras** (subenquadramento, Lei 14.133/2021).

## Princípio metodológico (PU Learning)

Os rótulos `engenharia` e `obras` da coleta são **confiáveis** — quando o órgão
marcou assim, é porque é. Só o rótulo `geral` é **ruidoso**: parte é serviço
comum de verdade, parte é engenharia mal classificada. O estudo usa os
confirmados como âncora para encontrar engenharia escondida no `geral`.

## Roteiro

| Etapa | O que faz |
|---|---|
| 0 | Setup + relatório vivo |
| 1 | Carregar base + pré-processamento + EDA mínima |
| 2 | Embeddings SBERT + **filtro PU** (centróide eng+obras) |
| 3 | **Clusterização auto-k** (6–12, escolhe o mais coeso) |
| 4 | **Revisão humana** assistida por LLM (você decide) |
| 5 | LLM descreve perfis eng vs não-eng |
| 6 | Treina classificador (LogReg + Random Forest) |
| 7 | **Pontua TODOS os `geral`** → ranking de suspeitos |
| 8 | Validação manual (~80 contratos) → precisão/recall |
| 9 | Visualização UMAP 2D |
| 10 | Veredito LLM nos suspeitos |
| 11 | **Análise de rito** (TR/Edital da licitação → ART/CREA) |
| 12 | Exportação (modelo + relatório) |

**Autossuficiente:** todo o código está neste notebook. Não precisa clonar nem
instalar pacote nenhum além das libs públicas. Apenas `contratos.parquet` no
Drive.

## 0. Setup

Bibliotecas, Drive, LLM e o **relatório vivo** (cada etapa atualiza um `.md` no
Drive automaticamente — você sempre tem o estado atual).

In [ ]:
!pip install -q sentence-transformers scikit-learn umap-learn nltk
!pip install -q google-generativeai pymupdf pyarrow pandas matplotlib seaborn

In [ ]:
import os, re, time, json, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import datetime as _dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for _p in ('punkt', 'punkt_tab', 'stopwords', 'rslp'):
    nltk.download(_p, quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score,
                              silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 160)
print('imports OK')

In [ ]:
# Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

CAMINHO_PARQUET  = '/content/drive/MyDrive/PNCP_TCC/dados/coleta/contratos.parquet'
PASTA_RESULTADOS = '/content/drive/MyDrive/PNCP_TCC/resultados_pesquisa'
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
print('Resultados em:', PASTA_RESULTADOS)

In [ ]:
# ── LLM: Google Gemini (auto-detecção + retry em rate limit) ────────────
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

_PREFERIDOS = ['gemini-2.5-flash-lite', 'gemini-2.0-flash-lite',
                'gemini-2.0-flash', 'gemini-2.5-flash', 'gemini-flash-latest']

def detectar_modelo():
    try:
        disp = [m.name.split('/')[-1] for m in genai.list_models()
                if 'generateContent' in getattr(m, 'supported_generation_methods', [])]
    except Exception:
        return 'gemini-2.0-flash'
    for p in _PREFERIDOS:
        if p in disp:
            return p
    return next((m for m in disp if 'flash' in m), 'gemini-2.0-flash')

MODELO_LLM = detectar_modelo()
INTERVALO_LLM_SEG = 4    # pausa entre chamadas (free tier ~15-30 RPM)

def llm_task(system, prompt, max_tokens=800, temperatura=0.2, max_tentativas=3):
    for tent in range(max_tentativas):
        try:
            gm = genai.GenerativeModel(MODELO_LLM,
                generation_config={'temperature': temperatura,
                                   'max_output_tokens': max_tokens})
            r = gm.generate_content([system, prompt])
            return (r.text or '').strip()
        except Exception as e:
            msg = str(e)
            if ('429' in msg or 'quota' in msg.lower()) and tent < max_tentativas - 1:
                time.sleep(20 * (2 ** tent))
                continue
            print(f'[llm] erro: {msg[:120]}')
            return None
    return None

def extrair_json(txt):
    if not txt:
        return None
    t = txt.replace('```json', '').replace('```', '')
    m = re.search(r'\{.*\}', t, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

print('LLM:', MODELO_LLM)

# ⚠ Alternativa SEM rate limit (Ollama local na GPU): descomente para usar.
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess; subprocess.Popen(['ollama','serve']); time.sleep(8)
# !ollama pull llama3.1
# import requests
# def llm_task(system, prompt, max_tokens=800, temperatura=0.2, **kw):
#     try:
#         r = requests.post('http://127.0.0.1:11434/api/chat', timeout=180,
#             json={'model':'llama3.1','stream':False,
#                   'messages':[{'role':'system','content':system},
#                               {'role':'user','content':prompt}],
#                   'options':{'temperature':temperatura,'num_predict':max_tokens}})
#         return r.json().get('message',{}).get('content','').strip() if r.ok else None
#     except Exception as e:
#         print('[llm]', e); return None
# INTERVALO_LLM_SEG = 0

In [ ]:
# ── Relatório vivo ──────────────────────────────────────────────────────
# Cada etapa chama add_md(secao, conteudo) e o .md é regravado no Drive.
RELATORIO_SECOES = {}
RELATORIO_ORDEM = []
RELATORIO_PATH = f'{PASTA_RESULTADOS}/relatorio_vivo.md'

def add_md(secao, conteudo):
    if secao not in RELATORIO_SECOES:
        RELATORIO_ORDEM.append(secao)
    RELATORIO_SECOES[secao] = conteudo
    linhas = ['# Pesquisa de Subenquadramento — Relatório',
              f'_Atualizado em {_dt.datetime.now():%Y-%m-%d %H:%M}_', '']
    for s in RELATORIO_ORDEM:
        linhas += [f'## {s}', RELATORIO_SECOES[s], '']
    Path(RELATORIO_PATH).write_text('\n'.join(linhas), encoding='utf-8')
    print('  → relatório vivo atualizado')

print('Relatório vivo:', RELATORIO_PATH)

## 1. Base + pré-processamento + EDA mínima

Carrega `contratos.parquet` e define as funções de texto (tudo inline). EDA só
o essencial: distribuição dos rótulos e top termos (para entender a base).

In [ ]:
# ── Funções de pré-processamento (inline) ───────────────────────────────
STOP_PT = set(nltk.corpus.stopwords.words('portuguese'))
# Termos burocráticos que aparecem em quase todo contrato e não discriminam.
STOP_DOMINIO = {
    'contratacao','contratacoes','contratar','contrato','contratos','servico',
    'servicos','prestacao','prestar','prestados','prestada','empresa','empresas',
    'especializada','especializado','fornecimento','fornecer','aquisicao',
    'aquisicoes','fornecedor','registro','preco','precos','atender','atendimento',
    'executar','execucao','realizacao','realizar','objeto','objetos','conforme',
    'demanda','demandas','referente','pregao','licitacao','edital','editais',
    'ata','atas','municipio','municipal','estadual','federal','lote','lotes',
    'item','itens',
}
STOP_TUDO = STOP_PT | STOP_DOMINIO
STEMMER = RSLPStemmer()

def _sem_acento(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s).lower())
                    if not unicodedata.combining(c))

def meu_tokenizador(doc):
    toks = word_tokenize(_sem_acento(doc), language='portuguese')
    return [STEMMER.stem(w) for w in toks
            if w not in STOP_TUDO and w.isalnum() and not w.isdigit() and len(w) > 2]

# Remove prefixos burocráticos antes do SBERT — sem isso, "CONTRATAÇÃO DE
# EMPRESA ESPECIALIZADA PARA PRESTAÇÃO DE SERVIÇOS DE..." idêntico em quase
# todo contrato arrasta a similaridade para cima e arruína a discriminação.
_RX_BP = [re.compile(p, re.IGNORECASE) for p in [
    r'^\s*registro\s+de\s+pre[çc]os?\s+(para\s+a?\s*)?',
    r'^\s*contrata[çc][ãa]o\s+(de\s+)?(empresa\s+)?(especializada\s+)?'
    r'(em\s+|para\s+(o?\s+|a?\s+)?(presta[çc][ãa]o\s+(de\s+)?)?(servi[çc]os?\s+(de\s+)?)?)?',
    r'^\s*contrata[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*presta[çc][ãa]o\s+de\s+servi[çc]os?\s+(de\s+)?',
    r'^\s*aquisi[çc][ãa]o\s+de\s+', r'^\s*fornecimento\s+de\s+',
    r'^\s*execu[çc][ãa]o\s+de\s+(servi[çc]os?\s+(de\s+)?)?',
    r'^\s*loca[çc][ãa]o\s+de\s+',
]]
def limpar_boilerplate(t):
    if not isinstance(t, str):
        return ''
    s = t.strip()
    for _ in range(3):
        for rx in _RX_BP:
            s = rx.sub('', s, count=1)
    return re.sub(r'\s+', ' ', s).strip()

print('funções de texto definidas')

In [ ]:
# ── Carrega base ────────────────────────────────────────────────────────
df_full = pd.read_parquet(CAMINHO_PARQUET)
COL_UF = next((c for c in df_full.columns
               if c.lower() in ('uf','ufsigla','siglauf','unidadefederativa')), None)
if COL_UF and COL_UF != 'uf':
    df_full = df_full.rename(columns={COL_UF: 'uf'})

cols = [c for c in ['numeroControlePNCP','objeto','rotulo','razaoSocialOrgao',
                     'municipioNome','valor','uf','numeroControlePNCPCompra',
                     'sequencialCompra','anoCompra'] if c in df_full.columns]
df = df_full[cols].copy().rename(columns={'objeto': 'text'})
del df_full
df = df.dropna(subset=['text'])
df = df[df['text'].str.len() > 20].reset_index(drop=True)

# Amostragem opcional (em GPU paga, suba ou use tudo)
N_AMOSTRA = 30_000
if len(df) > N_AMOSTRA:
    df = df.sample(n=N_AMOSTRA, random_state=42).reset_index(drop=True)

print(f'Base de trabalho: {len(df):,} contratos')
print(df['rotulo'].value_counts().to_string())

In [ ]:
# ── EDA mínima: distribuição + top termos ───────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.2))
cont = df['rotulo'].value_counts()
cores = {'engenharia':'#d62728','obras':'#ff7f0e','geral':'#9aa0a6'}
ax.bar(cont.index, cont.values, color=[cores.get(c,'#1f77b4') for c in cont.index])
for c, v in zip(cont.index, cont.values):
    ax.text(c, v, f'{v:,}', ha='center', va='bottom')
ax.set_title(f'Distribuição dos rótulos (n={len(df):,})'); ax.set_ylabel('nº')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/01_distribuicao.png', dpi=120, bbox_inches='tight')
plt.show()

vsm = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                      min_df=5, ngram_range=(1,2), max_features=20000)
Xtf = vsm.fit_transform(df['text'])
top = (pd.DataFrame({'termo': vsm.get_feature_names_out(),
                     'peso': Xtf.toarray().sum(axis=0)})
       .sort_values('peso', ascending=False).head(25))
print('\nTop 25 termos/bigramas:')
print(top.to_string(index=False))

add_md('1. Base',
f'''- Contratos analisados: **{len(df):,}**
- Distribuição: {", ".join(f"{k}={v:,}" for k,v in cont.items())}
- Gráfico: `01_distribuicao.png`''')

## 2. Embeddings SBERT + filtro PU

**SBERT** transforma cada objeto em vetor semântico. Depois, usamos os
**confirmados (eng+obras)** como âncora: calculamos o centróide deles e medimos
a proximidade de cada `geral`. Só os `geral` próximos entram na análise — o
resto é claramente "não-engenharia".

- `eng+obras` → `rotulo_auto = 'eng_obra'` (positivos certeiros)
- `geral` próximo do centróide → **candidato** (vai para clusterização)
- `geral` distante → `rotulo_auto = 'nao'` (negativo confirmado automático)

In [ ]:
# 2.1 Embeddings
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('distiluse-base-multilingual-cased-v1')

print('Gerando embeddings...')
textos = df['text'].map(limpar_boilerplate).tolist()
X_emb = sbert.encode(textos, show_progress_bar=True, batch_size=64,
                     convert_to_numpy=True, normalize_embeddings=True)
print('Embeddings:', X_emb.shape)

In [ ]:
# 2.2 Filtro PU
mask_pos   = df['rotulo'].isin(['engenharia', 'obras']).values
mask_geral = (df['rotulo'] == 'geral').values

centroide = X_emb[mask_pos].mean(axis=0)
centroide = centroide / (np.linalg.norm(centroide) + 1e-12)
sim = X_emb @ centroide          # X_emb já normalizado → produto = cosseno
df['sim_centroide'] = sim

PCT_GERAL_CANDIDATO = 0.30       # top 30% dos geral mais próximos = candidatos
limiar = float(np.quantile(sim[mask_geral], 1 - PCT_GERAL_CANDIDATO))

df['eh_candidato'] = mask_pos | (mask_geral & (sim >= limiar))
df['rotulo_auto'] = ''
df.loc[mask_pos, 'rotulo_auto'] = 'eng_obra'
df.loc[mask_geral & (sim < limiar), 'rotulo_auto'] = 'nao'

n_pos = int(mask_pos.sum())
n_cand_g = int((mask_geral & (sim >= limiar)).sum())
n_nao = int((mask_geral & (sim < limiar)).sum())
print(f'Positivos certeiros (eng+obras): {n_pos:,}')
print(f'Candidatos do geral (próximos):  {n_cand_g:,}')
print(f'Geral → "nao" automático:        {n_nao:,}')
print(f'Limiar de similaridade: {limiar:.3f}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(sim[mask_pos], bins=50, alpha=.7, density=True, color='#1b7837',
        label=f'eng+obras (n={n_pos:,})')
ax.hist(sim[mask_geral], bins=50, alpha=.5, density=True, color='#9aa0a6',
        label=f'geral (n={int(mask_geral.sum()):,})')
ax.axvline(limiar, color='red', ls='--', lw=1.5,
           label=f'limiar (top {PCT_GERAL_CANDIDATO:.0%} dos geral)')
ax.set_xlabel('similaridade ao centróide eng+obras'); ax.set_ylabel('densidade')
ax.set_title('Filtro PU: confirmados vs. geral'); ax.legend()
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/02_filtro_pu.png', dpi=120, bbox_inches='tight')
plt.show()

add_md('2. Filtro PU',
f'''- Positivos certeiros (eng+obras): **{n_pos:,}**
- Candidatos do geral (top {PCT_GERAL_CANDIDATO:.0%}, próximos do centróide): **{n_cand_g:,}**
- Geral classificado como "nao" automaticamente: **{n_nao:,}**
- Limiar de similaridade: {limiar:.3f}
- Gráfico: `02_filtro_pu.png`''')

## 3. Clusterização auto-k (poucos e coesos)

KMeans sobre os **candidatos** (eng+obras + geral próximo). Testamos k=6..12 e
escolhemos o de melhor **silhouette** — garante poucos clusters e bem
separados. Cada cluster mostra o **% de positivos certeiros**: quanto maior,
mais o cluster "é" engenharia.

In [ ]:
# 3.1 Escolhe melhor k por silhouette
mask_cand = df['eh_candidato'].values
X_cand = X_emb[mask_cand]
idx_cand = np.where(mask_cand)[0]
print(f'Candidatos a clusterizar: {len(idx_cand):,}')

rng = np.random.RandomState(42)
amostra_sil = rng.choice(len(X_cand), size=min(5000, len(X_cand)), replace=False)

resultados_k = []
for k in range(6, 13):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(X_cand)
    s = silhouette_score(X_cand[amostra_sil], lab[amostra_sil], metric='cosine')
    resultados_k.append((k, s, km, lab))
    print(f'  k={k}: silhouette={s:.3f}')

melhor_k, melhor_sil, melhor_km, melhor_lab = max(resultados_k, key=lambda r: r[1])
print(f'\n🏆 Melhor k = {melhor_k} (silhouette={melhor_sil:.3f})')

df['cluster'] = -1
df.loc[idx_cand, 'cluster'] = melhor_lab

In [ ]:
# 3.2 Descritores de cada cluster (TF-IDF) + pureza de positivos
df_cand = df[df['eh_candidato']].copy()
vsm_d = TfidfVectorizer(tokenizer=meu_tokenizador, token_pattern=None,
                        min_df=3, ngram_range=(1,2), max_features=30000)
Xd = vsm_d.fit_transform(df_cand['text'])
nomes = vsm_d.get_feature_names_out()

desc = []
for c in sorted(df_cand['cluster'].unique()):
    m = (df_cand['cluster'] == c).values
    sub = df_cand[m]
    soma = Xd[m].sum(axis=0).A1
    top_termos = ', '.join(nomes[i] for i in soma.argsort()[::-1][:8])
    d = sub['rotulo'].value_counts().to_dict()
    n_cert = d.get('engenharia',0) + d.get('obras',0)
    desc.append({'cluster': c, 'n_docs': len(sub), 'top_termos': top_termos,
                 'n_certeiros': n_cert, 'n_geral_cand': d.get('geral',0),
                 'pct_certeiros': n_cert/max(1,len(sub))})
df_desc = pd.DataFrame(desc).sort_values('pct_certeiros', ascending=False)
print('Clusters (ordenados por % de positivos certeiros):\n')
print(df_desc[['cluster','n_docs','n_certeiros','n_geral_cand',
               'pct_certeiros','top_termos']].to_string(index=False))

add_md('3. Clusterização',
f'''- Melhor k = **{melhor_k}** (silhouette {melhor_sil:.3f})
- Candidatos clusterizados: {len(df_cand):,}

| cluster | n | % certeiros | top termos |
|---|---|---|---|
''' + '\n'.join(
    f"| {r['cluster']} | {r['n_docs']} | {r['pct_certeiros']:.0%} | {r['top_termos'][:55]} |"
    for _, r in df_desc.iterrows()))

## 4. Revisão humana assistida por LLM

A LLM lê os top termos + amostras de cada cluster e **sugere** um rótulo
(`eng_obra`/`nao`/`duvidoso`) com justificativa. Isso vira a coluna
`sugestao_llm` no CSV. **Você (engenheiro) decide**: confirma ou corrige na
coluna `rotulo_humano`.

| Rótulo | Quando usar |
|---|---|
| `eng_obra` | Certeza de que o cluster é engenharia/obra |
| `nao` | Certeza de que NÃO é (limpeza, evento, fornecimento, etc.) |
| `duvidoso` | Ambíguo — o classificador (Etapa 6) decide cada caso |

In [ ]:
# 4.1 LLM sugere rótulo de cada cluster
SYS_CLUSTER = '''Você é engenheiro especialista em contratações públicas
(Lei 14.133/2021). Recebe os termos mais frequentes e exemplos de objetos de
um GRUPO de contratos. Classifique o grupo.

Considere apenas ENGENHARIA (exige CREA/ART) e OBRAS. Arquitetura (CAU/RRT)
NÃO entra. NÃO são engenharia (são "nao"): locação de equipamento, aquisição
de bens, eventos/cursos, serviços administrativos, fornecimento de energia/água.

Responda APENAS no JSON:
{"rotulo": "eng_obra"|"nao"|"duvidoso",
 "confianca": 0.0-1.0,
 "justificativa": "1 frase"}'''

sugestoes = {}
for c in sorted(df_cand['cluster'].unique()):
    sub = df_cand[df_cand['cluster'] == c]
    drow = df_desc[df_desc['cluster'] == c].iloc[0]
    amostras = '\n'.join(f'- {str(o)[:200]}'
                          for o in sub['text'].sample(min(10, len(sub)),
                                                       random_state=c))
    prompt = (f"Termos: {drow['top_termos']}\n\nExemplos:\n{amostras}\n\n"
              f"Classifique o grupo.")
    p = extrair_json(llm_task(SYS_CLUSTER, prompt)) or {}
    sugestoes[c] = p
    print(f"cluster {c}: {p.get('rotulo','?'):9s} "
          f"({p.get('confianca',0)}) — {p.get('justificativa','')[:70]}")
    time.sleep(INTERVALO_LLM_SEG)

In [ ]:
# 4.2 Gera CSV de revisão (você preenche rotulo_humano)
N_AMOSTRAS = 10
linhas = []
for c in sorted(df_cand['cluster'].unique()):
    sub = df_cand[df_cand['cluster'] == c]
    drow = df_desc[df_desc['cluster'] == c].iloc[0]
    sug = sugestoes.get(c, {})
    # amostras: mistura certeiros + candidatos geral
    pos = sub[sub['rotulo'].isin(['engenharia','obras'])]['text']
    ger = sub[sub['rotulo'] == 'geral']['text']
    amostras = (list(pos.sample(min(4, len(pos)), random_state=c)) +
                list(ger.sample(min(N_AMOSTRAS-4, len(ger)), random_state=c)))
    linha = {'cluster': c, 'n_docs': drow['n_docs'],
             'n_certeiros': drow['n_certeiros'],
             'n_geral_cand': drow['n_geral_cand'],
             'pct_certeiros': round(drow['pct_certeiros'], 3),
             'top_termos': drow['top_termos'],
             'sugestao_llm': sug.get('rotulo', ''),
             'conf_llm': sug.get('confianca', ''),
             'justif_llm': sug.get('justificativa', ''),
             'rotulo_humano': sug.get('rotulo', '')}   # pré-preenche c/ sugestão
    for i, a in enumerate(amostras, 1):
        linha[f'amostra_{i}'] = str(a)[:250]
    linhas.append(linha)

caminho_revisao = f'{PASTA_RESULTADOS}/04_revisao_clusters.csv'
pd.DataFrame(linhas).to_csv(caminho_revisao, index=False, encoding='utf-8-sig')
print(f'CSV criado: {caminho_revisao}')
print('\n→ ABRA no Excel/Sheets. A coluna rotulo_humano já vem com a SUGESTÃO')
print('  da LLM — REVISE e corrija onde discordar. Valores: eng_obra|nao|duvidoso')

### ⏸ PAUSE — revise o CSV `04_revisao_clusters.csv`, ajuste `rotulo_humano`, salve, e rode a célula abaixo

In [ ]:
# 4.3 Carrega revisão e propaga para o df
df_rev = pd.read_csv(caminho_revisao)
df_rev['rotulo_humano'] = df_rev['rotulo_humano'].fillna('').str.strip().str.lower()
validos = {'eng_obra', 'nao', 'duvidoso'}
inval = df_rev[~df_rev['rotulo_humano'].isin(validos)]
if len(inval):
    print(f'⚠ {len(inval)} cluster(s) sem rótulo válido: '
          f'{inval[["cluster","rotulo_humano"]].values.tolist()}')
    print('Preencha e rode de novo.')
else:
    mapa = dict(zip(df_rev['cluster'], df_rev['rotulo_humano']))
    # rotulo_humano: parte do rotulo_auto; candidatos geral usam veredito do cluster
    df['rotulo_humano'] = df['rotulo_auto']
    mcg = (df['rotulo'] == 'geral') & df['eh_candidato']
    df.loc[mcg, 'rotulo_humano'] = df.loc[mcg, 'cluster'].map(mapa)
    cont = df['rotulo_humano'].value_counts()
    print('Distribuição final dos rótulos:')
    print(cont.to_string())
    conc = (df_rev['sugestao_llm'] == df_rev['rotulo_humano']).mean()
    add_md('4. Revisão humana',
f'''- Clusters revisados: {len(df_rev)}
- Concordância LLM × humano: {conc:.0%}
- Distribuição final: {", ".join(f"{k}={v:,}" for k,v in cont.items())}

{df_rev[["cluster","pct_certeiros","sugestao_llm","rotulo_humano"]].to_markdown(index=False)}''')

## 5. LLM descreve os perfis (documentação)

A partir dos clusters certeiros, a LLM sintetiza o perfil de cada classe —
padrões léxicos e tipos de serviço. Serve para documentar e auditar.

In [ ]:
SYS_PERFIL = '''Você é engenheiro analista de contratações públicas. Recebe
objetos de contratos de um grupo. Descreva o perfil em JSON:
{"resumo": "1-2 frases",
 "padroes_lexicais": ["termo1","termo2","termo3"],
 "tipos_servico": ["tipo1","tipo2"]}'''

def perfil(rot, n=15):
    sub = df[df['rotulo_humano'] == rot]
    if not len(sub):
        return {}
    objs = '\n'.join(f'- {str(o)[:200]}'
                      for o in sub['text'].sample(min(n, len(sub)), random_state=1))
    return extrair_json(llm_task(SYS_PERFIL,
                                 f'Grupo "{rot}":\n{objs}\n\nDescreva.')) or {}

perfil_eng = perfil('eng_obra'); time.sleep(INTERVALO_LLM_SEG)
perfil_nao = perfil('nao')
print('ENG_OBRA:', json.dumps(perfil_eng, ensure_ascii=False, indent=2))
print('\nNAO:', json.dumps(perfil_nao, ensure_ascii=False, indent=2))
json.dump({'eng_obra': perfil_eng, 'nao': perfil_nao},
          open(f'{PASTA_RESULTADOS}/05_perfis.json', 'w'), ensure_ascii=False, indent=2)
add_md('5. Perfis (LLM)',
f'''**eng_obra:** {perfil_eng.get("resumo","-")}
- léxico: {", ".join(perfil_eng.get("padroes_lexicais",[])[:5])}

**nao:** {perfil_nao.get("resumo","-")}
- léxico: {", ".join(perfil_nao.get("padroes_lexicais",[])[:5])}''')

## 6. Treina o classificador (LogReg + Random Forest)

**Treino:** positivos = eng+obras + clusters `eng_obra`; negativos = geral
distante + clusters `nao`. (Os `duvidoso` ficam de fora — serão pontuados na
Etapa 7.) Features = embeddings SBERT. Comparamos os dois modelos por F1 e
escolhemos o melhor.

In [ ]:
# 6.1 Monta treino (certeiros: eng_obra vs nao)
mask_tr = df['rotulo_humano'].isin(['eng_obra', 'nao']).values
X_tr_all = X_emb[mask_tr]
y_tr_all = (df.loc[mask_tr, 'rotulo_humano'] == 'eng_obra').astype(int).values
print(f'Treino: {mask_tr.sum():,} | eng_obra={y_tr_all.sum():,} | nao={(y_tr_all==0).sum():,}')

Xa, Xb, ya, yb = train_test_split(X_tr_all, y_tr_all, test_size=0.2,
                                   random_state=42, stratify=y_tr_all)

modelos = {
    'logreg': LogisticRegression(max_iter=3000, class_weight='balanced',
                                  C=1.0, n_jobs=-1, random_state=42),
    'rf': RandomForestClassifier(n_estimators=200, max_depth=20,
                                  class_weight='balanced', n_jobs=-1, random_state=42),
}
res = {}
for nome, m in modelos.items():
    m.fit(Xa, ya)
    pred = m.predict(Xb)
    res[nome] = {'modelo': m, 'f1': f1_score(yb, pred, average='macro'),
                 'pred': pred}
    print(f'  {nome}: F1 macro = {res[nome]["f1"]:.3f}')

melhor_nome = max(res, key=lambda n: res[n]['f1'])
melhor = res[melhor_nome]['modelo']
print(f'\n🏆 Melhor: {melhor_nome} (F1={res[melhor_nome]["f1"]:.3f})')

In [ ]:
# 6.2 Matriz de confusão
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nome, r) in zip(axes, res.items()):
    cm = confusion_matrix(yb, r['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['nao','eng_obra'], yticklabels=['nao','eng_obra'])
    ax.set_title(f'{nome} (F1={r["f1"]:.3f})'); ax.set_xlabel('predito'); ax.set_ylabel('real')
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/06_confusao.png', dpi=120, bbox_inches='tight')
plt.show()

add_md('6. Classificador',
'| modelo | F1 macro |\n|---|---|\n' +
'\n'.join(f'| {n} | {r["f1"]:.3f} |' for n, r in res.items()) +
f'\n\n🏆 Melhor: **{melhor_nome}**')

## 7. Pontua TODOS os `geral` → ranking de suspeitos (produto final)

Aplica o melhor modelo a **todos** os contratos `geral` (inclusive os que o
filtro PU descartou) e gera a probabilidade de ser engenharia. Esse ranking é
o produto central da pesquisa: a lista priorizada de prováveis
subenquadramentos.

In [ ]:
mask_g = (df['rotulo'] == 'geral').values
prob_g = melhor.predict_proba(X_emb[mask_g])[:, 1]
df['prob_eng_obra'] = np.nan
df.loc[mask_g, 'prob_eng_obra'] = prob_g

df['classe_final'] = df['rotulo_humano']
df.loc[mask_g, 'classe_final'] = np.where(prob_g >= 0.5, 'eng_obra_pred', 'nao_pred')

ranking = (df[mask_g].sort_values('prob_eng_obra', ascending=False)
           [['numeroControlePNCP','text','cluster','prob_eng_obra','valor']])
ranking.to_csv(f'{PASTA_RESULTADOS}/07_ranking_suspeitos.csv',
               index=False, encoding='utf-8-sig')

n_susp = int((prob_g >= 0.5).sum())
n_alta = int((prob_g >= 0.8).sum())
print(f'Geral pontuados: {mask_g.sum():,}')
print(f'Suspeitos (prob >= 0.5): {n_susp:,}')
print(f'Alta confiança (prob >= 0.8): {n_alta:,}')
print('\nTop 15 suspeitos:')
print(ranking.head(15)[['numeroControlePNCP','text','prob_eng_obra']].to_string(index=False))

add_md('7. Ranking de suspeitos',
f'''- Geral pontuados: **{int(mask_g.sum()):,}**
- Suspeitos (prob ≥ 0.5): **{n_susp:,}**
- Alta confiança (prob ≥ 0.8): **{n_alta:,}**
- CSV: `07_ranking_suspeitos.csv`''')

## 8. Validação manual (rigor acadêmico)

Sorteamos ~80 `geral` estratificados por faixa de probabilidade para você
rotular à mão. Comparando com o modelo, obtemos **precisão/recall/F1 honestos**
— sem isso, não há número confiável de desempenho para o TCC.

In [ ]:
# 8.1 Gera amostra de validação estratificada por faixa de prob
N_VALID = 80
g = df[mask_g].copy()
g['faixa'] = pd.cut(g['prob_eng_obra'], bins=[0,.2,.4,.6,.8,1.0],
                    labels=['0-20','20-40','40-60','60-80','80-100'])
val = g.groupby('faixa', group_keys=False, observed=True).apply(
    lambda x: x.sample(min(len(x), N_VALID//5), random_state=42))
val = val[['numeroControlePNCP','text','prob_eng_obra','faixa']].copy()
val['rotulo_verdade'] = ''     # você preenche: eng_obra | nao
cam_val = f'{PASTA_RESULTADOS}/08_validacao.csv'
val.to_csv(cam_val, index=False, encoding='utf-8-sig')
print(f'Amostra de validação: {len(val)} contratos → {cam_val}')
print('→ Preencha rotulo_verdade (eng_obra|nao) e rode a próxima célula.')

### ⏸ PAUSE — rotule `08_validacao.csv` (coluna `rotulo_verdade`), salve, e rode abaixo

In [ ]:
# 8.2 Mede precisão/recall/F1 contra a verdade manual
vdf = pd.read_csv(cam_val)
vdf['rotulo_verdade'] = vdf['rotulo_verdade'].fillna('').str.strip().str.lower()
vdf = vdf[vdf['rotulo_verdade'].isin(['eng_obra','nao'])]
if len(vdf) < 10:
    print('⚠ Poucas linhas rotuladas. Preencha rotulo_verdade e rode de novo.')
else:
    y_true = (vdf['rotulo_verdade'] == 'eng_obra').astype(int)
    y_pred = (vdf['prob_eng_obra'] >= 0.5).astype(int)
    P = precision_score(y_true, y_pred, zero_division=0)
    R = recall_score(y_true, y_pred, zero_division=0)
    F = f1_score(y_true, y_pred, zero_division=0)
    print(f'Validação manual ({len(vdf)} contratos):')
    print(f'  Precisão (dos preditos eng, quantos são eng): {P:.1%}')
    print(f'  Recall   (dos eng reais, quantos pegou):      {R:.1%}')
    print(f'  F1: {F:.3f}')
    print('\n' + classification_report(y_true, y_pred,
          target_names=['nao','eng_obra'], zero_division=0))
    add_md('8. Validação manual',
f'''- Contratos rotulados à mão: {len(vdf)}
- **Precisão: {P:.1%}** | **Recall: {R:.1%}** | **F1: {F:.3f}**''')

## 9. Visualização UMAP 2D

Projeta os embeddings em 2D. Cores por `classe_final`. Se os `geral` preditos
como engenharia (verde claro) caem junto dos certeiros eng (verde escuro), o
modelo concorda geometricamente.

In [ ]:
import umap
MAX_VIZ = 8000
idx_v = (np.random.RandomState(42).choice(len(df), MAX_VIZ, replace=False)
         if len(df) > MAX_VIZ else np.arange(len(df)))
print('Projetando UMAP...')
emb2d = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine',
                  random_state=42).fit_transform(X_emb[idx_v])

dv = df.iloc[idx_v].copy()
dv['x'], dv['y'] = emb2d[:,0], emb2d[:,1]
paleta = {'eng_obra':'#1b7837','nao':'#b2182b',
          'eng_obra_pred':'#7fbc41','nao_pred':'#ef8a62',
          'duvidoso':'#cccccc'}
fig, ax = plt.subplots(figsize=(11, 8))
for cl, cor in paleta.items():
    s = dv[dv['classe_final'] == cl]
    if len(s):
        ax.scatter(s['x'], s['y'], c=cor, s=8, alpha=.55,
                   label=f'{cl} ({len(s)})', edgecolors='none')
ax.legend(); ax.set_title(f'UMAP 2D — classe final (modelo: {melhor_nome})')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(f'{PASTA_RESULTADOS}/09_umap.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Veredito LLM nos suspeitos

A LLM revisa os top suspeitos (regra de ouro: locação/aquisição/evento = não)
como dupla checagem antes da análise de rito.

In [ ]:
SYS_VER = '''Você é engenheiro analista (Lei 14.133/2021). Classifique o TIPO
do contrato pelo objeto. Só engenharia (CREA/ART) e obras contam. NÃO contam
(são "nao"): locação de equipamento, aquisição de bens, evento/curso, serviços
administrativos, fornecimento de energia/água. Engenharia/obras = EXECUÇÃO de
serviço técnico ou intervenção física.
Responda JSON: {"classe":"eng_obra"|"nao","confianca":0.0-1.0,"motivo":"até 30 palavras"}'''

N_VEREDITO = 40
top_susp = df[mask_g].sort_values('prob_eng_obra', ascending=False).head(N_VEREDITO)
ver = []
for i, (_, row) in enumerate(top_susp.iterrows()):
    if i: time.sleep(INTERVALO_LLM_SEG)
    p = extrair_json(llm_task(SYS_VER, f"Objeto: {str(row['text'])[:500]}")) or {}
    ver.append({'numeroControlePNCP': row['numeroControlePNCP'],
                'prob_modelo': round(float(row['prob_eng_obra']), 3),
                'llm_classe': p.get('classe','?'),
                'llm_conf': float(p.get('confianca',0) or 0),
                'llm_motivo': str(p.get('motivo',''))[:200],
                'objeto': str(row['text'])[:160]})
    if (i+1) % 10 == 0: print(f'  {i+1}/{len(top_susp)}')

df_ver = pd.DataFrame(ver)
df_ver['confirmado'] = (df_ver['llm_classe']=='eng_obra') & (df_ver['llm_conf']>=0.6)
n_conf = int(df_ver['confirmado'].sum())
df_ver.to_csv(f'{PASTA_RESULTADOS}/10_veredito_llm.csv', index=False, encoding='utf-8-sig')
print(f'\nLLM confirmou {n_conf}/{len(df_ver)} como engenharia.')
add_md('10. Veredito LLM', f'- Top {len(df_ver)} suspeitos revisados → **{n_conf}** confirmados como engenharia')
df_ver.head(20)

## 11. Análise de RITO (evidência definitiva)

Para os suspeitos **confirmados pelo LLM**, baixamos os documentos da
**licitação (compra)** — é onde ficam o TR, Projeto Básico e Edital (o contrato
em si não os expõe). Detectamos marcadores legais (ART/CREA, projeto básico,
ABNT, etc.) e pedimos um veredito à LLM sobre o trecho do TR.

**Descoberta-chave (Manual API PNCP §4.1):** o `numeroControlePNCP` tem dígito
1 = contratação (compra) e 2 = contrato. O TR/PB pertencem à compra,
referenciada pelo campo `numeroControlePNCPCompra`.

**Veredito final:**
- `subenquadramento_real` — PDF legível SEM rito → provável violação da Lei 14.133
- `rotulacao_incorreta_processo_ok` — rito presente → erro de cadastro
- `indeterminado_*` — compra não resolvida / sem documento / PDF ilegível

Tudo inline (sem dependências externas além de `requests` + PyMuPDF).

In [ ]:
# 11.1 Funções inline de download + marcadores (copiadas do pipeline validado)
import requests

MAX_PDFS_BAIXAR = 50          # limite de contratos (cada um leva ~5-20s)
MAX_DOCS_POR_CONTRATO = 3
CACHE_PDFS = Path(PASTA_RESULTADOS) / 'cache_pdfs'
CACHE_PDFS.mkdir(parents=True, exist_ok=True)
_API = 'https://pncp.gov.br/api/pncp/v1'
_HEADERS = {'User-Agent': 'Mozilla/5.0 (pesquisa academica PNCP)'}
_RX_NCP = re.compile(r'^(?P<cnpj>\d{14})-(?P<tipo>\d+)-(?P<seq>\d+)/(?P<ano>\d{4})$')

def _decompor_ncp(num):
    m = _RX_NCP.match(str(num).strip()) if num else None
    return ({'cnpj': m['cnpj'], 'tipo': int(m['tipo']),
             'ano': int(m['ano']), 'sequencial': int(m['seq'])} if m else None)

# Marcadores de RITO DE ENGENHARIA — apenas CREA/ART (sem CAU/RRT)
MARCADORES = {
    'ART': [r'\banota[çc][ãa]o\s+de\s+responsabilidade\s+t[ée]cnica\b',
            r'\bART\b(?:\s+do\s+CREA)?'],
    'CREA': [r'\bCREA[/\s\-]?\w{0,2}\b',
             r'\bConselho\s+Regional\s+de\s+Engenharia\b'],
    'ENGENHEIRO_RESP': [r'\bengenheir[oa]?\s+respons[áa]vel\b',
                         r'\brespons[áa]vel\s+t[ée]cnico\b'],
    'ATESTADO_CAP_TEC': [r'\batestado\s+de\s+capacidade\s+t[ée]cnica\b'],
    'PROJETO_BASICO': [r'\bprojeto\s+b[áa]sico\b', r'\bprojeto\s+executivo\b',
                        r'\bmemorial\s+descritivo\b'],
    'OBRA_SERV_ENG': [r'\bobra\s+de\s+engenharia\b',
                       r'\bservi[çc]os?\s+de\s+engenharia\b'],
    'ABNT_NORMA': [r'\bABNT\s+NBR\s*\d+'],
    'LEI_14133_ENG': [r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XII\b',
                       r'\bart\.?\s*6[°º]?,?\s*(?:inc\.?\s*)?XX(?:I+)?\b'],
}
_RX_MARC = {n: [re.compile(p, re.IGNORECASE) for p in ps] for n, ps in MARCADORES.items()}

def detectar_marcadores(texto):
    if not texto:
        return {n: False for n in MARCADORES}
    return {n: any(rx.search(texto) for rx in lst) for n, lst in _RX_MARC.items()}

def normalizar_pdf_text(t):
    if not t:
        return ''
    t = re.sub(r'-\s*\n\s*', '', t)
    t = re.sub(r'(?<!\n)\n(?!\n)', ' ', t)
    return re.sub(r'[ \t]+', ' ', t)

def extrair_texto_pdf(caminho, max_pag=30):
    try:
        import fitz
        doc = fitz.open(caminho)
        txt = []
        for i, p in enumerate(doc):
            if i >= max_pag:
                break
            txt.append(p.get_text())
        doc.close()
        return normalizar_pdf_text('\n'.join(txt))
    except Exception:
        return ''

def _resolver_compra(ncp_contrato, row):
    nc = row.get('numeroControlePNCPCompra')
    if isinstance(nc, str) and nc.strip():
        info = _decompor_ncp(nc)
        if info:
            return info
    info_c = _decompor_ncp(ncp_contrato)
    if not info_c:
        return None
    if info_c['tipo'] == 1:
        return info_c
    try:
        r = requests.get(f'{_API}/orgaos/{info_c["cnpj"]}/contratos/'
                         f'{info_c["ano"]}/{info_c["sequencial"]}',
                         timeout=20, headers=_HEADERS)
        if r.status_code == 200:
            d = r.json()
            if d.get('numeroControlePNCPCompra'):
                return _decompor_ncp(d['numeroControlePNCPCompra'])
            sq, an = d.get('sequencialCompra'), d.get('anoCompra')
            cj = (d.get('orgaoEntidade') or {}).get('cnpj') or info_c['cnpj']
            if sq and an:
                return {'cnpj': cj, 'tipo': 1, 'ano': int(an), 'sequencial': int(sq)}
    except Exception:
        return None
    return None

def _listar_docs(info):
    try:
        r = requests.get(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                         f'{info["ano"]}/{info["sequencial"]}/arquivos',
                         timeout=20, headers=_HEADERS)
        if r.status_code != 200:
            return []
        d = r.json()
        return d if isinstance(d, list) else d.get('data', [])
    except Exception:
        return []

def _baixar(d, info, destino):
    urls = [d[c] for c in ('url','uri','urlArquivo','link') if d.get(c)]
    sq = (d.get('sequencialDocumento') or d.get('sequencial') or d.get('id'))
    if sq:
        urls.append(f'{_API}/orgaos/{info["cnpj"]}/compras/'
                    f'{info["ano"]}/{info["sequencial"]}/arquivos/{sq}')
    for u in urls:
        try:
            r = requests.get(u, timeout=60, stream=True, headers=_HEADERS)
            if r.status_code == 200:
                with open(destino, 'wb') as f:
                    for ch in r.iter_content(8192):
                        f.write(ch)
                if destino.stat().st_size > 0:
                    return True
        except Exception:
            continue
    return False

print('funções de rito definidas')

In [ ]:
# 11.2 Baixa e analisa o rito dos confirmados
SYS_RITO = '''Você analisa um TRECHO de Termo de Referência / Projeto Básico /
Edital de licitação pública. Diga se evidencia RITO DE ENGENHARIA: exigência de
ART do CREA, projeto básico/executivo, engenheiro responsável, atestado de
capacidade técnica, memorial descritivo, planilha orçamentária, normas ABNT, ou
artigos 6º XII/XX/XXI da Lei 14.133/2021.
Responda JSON: {"rito":"sim"|"nao"|"parcial","evidencias":["..."],"confianca":0.0-1.0}'''

conf_ncp = df_ver[df_ver['confirmado']]['numeroControlePNCP'].astype(str).tolist()
conf_ncp = conf_ncp[:MAX_PDFS_BAIXAR]
df_i = df.set_index(df['numeroControlePNCP'].astype(str))
print(f'Analisando rito de {len(conf_ncp)} confirmados...')

rito = []
n_baix = n_diag = 0
for i, ncp in enumerate(conf_ncp):
    row = df_i.loc[ncp]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    reg = {'numeroControlePNCP': ncp, 'objeto': str(row['text'])[:200],
           'prob_eng_obra': round(float(row.get('prob_eng_obra') or 1.0), 3),
           'ncp_compra': '', 'n_pdfs': 0, 'chars': 0, 'mk_score': 0,
           'llm_rito': '', 'llm_conf': 0.0}
    info_c = _resolver_compra(ncp, row)
    if not info_c:
        reg['classificacao_rito'] = 'indeterminado_sem_compra'
        rito.append(reg); continue
    reg['ncp_compra'] = f'{info_c["cnpj"]}-1-{info_c["sequencial"]:06d}/{info_c["ano"]}'
    docs = _listar_docs(info_c)
    if n_diag < 5:
        print(f'  • {ncp} → compra {reg["ncp_compra"]} docs={len(docs)}'); n_diag += 1
    if not docs:
        reg['classificacao_rito'] = 'indeterminado_sem_documento'
        rito.append(reg); continue
    textos = []
    for d in docs[:MAX_DOCS_POR_CONTRATO]:
        sq = d.get('sequencialDocumento') or d.get('sequencial') or d.get('id') or len(textos)
        dest = CACHE_PDFS / f'{reg["ncp_compra"].replace("/","_")}_{sq}.pdf'
        if not (dest.exists() and dest.stat().st_size > 0):
            if not _baixar(d, info_c, dest):
                continue
            n_baix += 1; time.sleep(0.2)
        textos.append(extrair_texto_pdf(dest))
    texto = '\n'.join(t for t in textos if t)
    reg['n_pdfs'] = len(textos); reg['chars'] = len(texto)
    if len(texto) < 300:
        reg['classificacao_rito'] = 'indeterminado_pdf_ilegivel'
        rito.append(reg); continue
    marc = detectar_marcadores(texto)
    for c, v in marc.items():
        reg[f'mk_{c}'] = bool(v)
    reg['mk_score'] = sum(1 for v in marc.values() if v)
    time.sleep(INTERVALO_LLM_SEG)
    pr = extrair_json(llm_task(SYS_RITO, f'Documento:\n{texto[:5000]}')) or {}
    reg['llm_rito'] = pr.get('rito', ''); reg['llm_conf'] = float(pr.get('confianca',0) or 0)
    if reg['mk_score'] >= 2 or pr.get('rito') == 'sim':
        reg['classificacao_rito'] = 'rotulacao_incorreta_processo_ok'
    else:
        reg['classificacao_rito'] = 'subenquadramento_real'
    rito.append(reg)

df_rito = pd.DataFrame(rito)
df_rito.to_csv(f'{PASTA_RESULTADOS}/11_analise_rito.csv', index=False, encoding='utf-8-sig')
print(f'\n{len(df_rito)} analisados | PDFs baixados={n_baix}')
print(df_rito['classificacao_rito'].value_counts().to_string())

In [ ]:
# 11.3 Gráfico do veredito de rito
if len(df_rito):
    fig, ax = plt.subplots(figsize=(8, 4))
    cont = df_rito['classificacao_rito'].value_counts()
    cr = {'subenquadramento_real':'#d62728','rotulacao_incorreta_processo_ok':'#ff7f0e',
          'indeterminado_sem_compra':'#9aa0a6','indeterminado_sem_documento':'#bdbdbd',
          'indeterminado_pdf_ilegivel':'#dddddd'}
    ax.barh(cont.index.astype(str)[::-1], cont.values[::-1],
            color=[cr.get(c,'#1f77b4') for c in cont.index[::-1]])
    for i, v in enumerate(cont.values[::-1]):
        ax.text(v, i, f' {v}', va='center')
    ax.set_title('Veredito de rito'); ax.set_xlabel('nº contratos')
    plt.tight_layout()
    fig.savefig(f'{PASTA_RESULTADOS}/11_rito.png', dpi=120, bbox_inches='tight')
    plt.show()
    n_viol = int((df_rito['classificacao_rito']=='subenquadramento_real').sum())
    print(f'⚠ {n_viol} prováveis violações da Lei 14.133/2021')
    add_md('11. Análise de rito',
f'''- Confirmados analisados: {len(df_rito)} | PDFs baixados: {n_baix}
- **Prováveis violações (subenquadramento_real): {n_viol}**
- Distribuição: {", ".join(f"{k}={v}" for k,v in cont.items())}''')

## 12. Exportação

Salva o **modelo pronto** (`modelo_final.joblib`), os CSVs e o relatório final.
A última célula mostra como classificar um contrato novo.

In [ ]:
import joblib
joblib.dump(melhor, f'{PASTA_RESULTADOS}/modelo_final.joblib')
df[['numeroControlePNCP','text','rotulo','cluster','rotulo_humano',
    'classe_final','prob_eng_obra']].to_csv(
    f'{PASTA_RESULTADOS}/12_df_classificado.csv', index=False, encoding='utf-8-sig')
print('Salvos: modelo_final.joblib, 12_df_classificado.csv')
print('Relatório vivo final:', RELATORIO_PATH)
print('\nArtefatos em', PASTA_RESULTADOS, ':')
for f in sorted(os.listdir(PASTA_RESULTADOS)):
    print('  -', f)

In [ ]:
# Reuso do modelo em contratos futuros
def classificar_objeto(texto):
    emb = sbert.encode([limpar_boilerplate(texto)], normalize_embeddings=True)
    pred = int(melhor.predict(emb)[0])
    prob = float(melhor.predict_proba(emb)[0, 1])
    return ('eng_obra' if pred == 1 else 'nao'), round(prob, 3)

for ex in ['CONTRATAÇÃO DE EMPRESA PARA PAVIMENTAÇÃO ASFÁLTICA DA RUA X',
           'AQUISIÇÃO DE MATERIAL DE ESCRITÓRIO',
           'PRESTAÇÃO DE SERVIÇO DE MANUTENÇÃO ELÉTRICA PREDIAL']:
    print(classificar_objeto(ex), '←', ex[:55])

---
### Resumo do método

1. **SBERT** representa cada objeto semanticamente
2. **Filtro PU**: eng+obras (confirmados) ancoram a busca; só `geral` próximo é investigado
3. **Clusters coesos** (auto-k) + **sua revisão** (assistida por LLM) geram rótulos limpos
4. **Classificador** (LogReg/RF) aprende com os rótulos limpos
5. **Todos os `geral`** recebem probabilidade → ranking de suspeitos
6. **Validação manual** dá precisão/recall honestos
7. **Rito** (TR/Edital da licitação) traz a evidência definitiva de violação

O `modelo_final.joblib` classifica contratos futuros direto do objeto.